# Báo cáo làm sạch dữ liệu bán hàng

Notebook này trình bày **kết quả làm sạch** từ `data/raw/` sang `data/processed/` bằng pipeline `scripts/clean_data.py` (logic tại `src/shared/data_cleaning.py`). Các bảng kết quả được đối chiếu trực tiếp từ số liệu sau khi chạy.

Các bảng nguồn có phạm vi và mục đích khác nhau (đơn hàng Superstore, RFM, Walmart, BigMart...). Không gộp các bảng khác grain thành một bảng chung. Nguồn Kaggle 2019–2020 được đánh dấu demo/unverified vì không có provenance giao dịch gốc được xác minh; tại thời điểm lập báo cáo file này chưa có trong `data/raw/`, nên phạm vi fact trong báo cáo là các nguồn đã thu thập đủ.

## 1. Phạm vi dữ liệu và phương pháp làm sạch

Nguồn đầu vào quét từ `data/raw/*.csv`; đầu ra đối chiếu tại `data/processed/*.csv`. Fact đơn hàng tuân thủ schema 10 cột nghiệp vụ + cột `Data Source` truy vết nguồn.

In [ ]:
from pathlib import Path
import tempfile

import pandas as pd
import plotly.express as px
from IPython.display import display

candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'data' / 'raw').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Không tìm thấy data/raw. '
        'Hãy mở notebook từ thư mục dự án.'
    )

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TMP_DIR = Path(tempfile.mkdtemp(prefix='clean_report_'))

from src.shared.data_cleaning import (
    REQUIRED_COLUMNS,
    build_monthly_sales,
    build_orders_union,
    build_rfm_export,
    clean_bigmart,
    clean_orders,
    clean_people,
    clean_walmart_features,
    clean_walmart_stores,
    clean_walmart_train,
    join_orders_people,
    join_walmart_weekly,
)

print(f'Thư mục dự án: {PROJECT_ROOT}')
print(f'Thư mục dữ liệu thô: {RAW_DIR}')
print(f'Thư mục dữ liệu đã xử lý: {PROCESSED_DIR}')
print(f'Schema fact chuẩn ({len(REQUIRED_COLUMNS)} cột): {REQUIRED_COLUMNS}')


## 2. Tổng quan nguồn dữ liệu thô

Bảng dưới đây tổng hợp quy mô và dung lượng các file hiện có trong `data/raw/`. Nguồn Kaggle 2019–2020 chưa được bàn giao nên chưa có mặt trong kỳ báo cáo này.

In [ ]:
inventory_rows = []
for path in sorted(RAW_DIR.glob('*.csv')):
    try:
        head = pd.read_csv(path, nrows=5)
        with open(path, encoding='utf-8', errors='ignore') as f:
            estimated_rows = sum(1 for _ in f) - 1
        inventory_rows.append({
            'File': path.name,
            'Số dòng (ước lượng)': estimated_rows,
            'Số cột': len(head.columns),
            'Dung lượng (MB)': round(path.stat().st_size / (1024 ** 2), 2),
            'Ví dụ cột': ', '.join(map(str, head.columns[:5])) + ' ...',
        })
    except Exception as exc:
        inventory_rows.append({'File': path.name, 'Số dòng (ước lượng)': 0, 'Số cột': 0, 'Dung lượng (MB)': 0.0, 'Ví dụ cột': f'Lỗi đọc: {exc}'})
inventory = pd.DataFrame(inventory_rows)
display(inventory)
print(f'Tổng số file raw trong kỳ báo cáo: {len(inventory)}.')
fig = px.bar(inventory, x='File', y='Số dòng (ước lượng)', title='Quy mô các file dữ liệu thô')
fig.update_layout(xaxis_title='File', yaxis_title='Số dòng')
fig.show()

missing_demo = RAW_DIR / 'global_electronics_retail_2019_2020.csv'
if not missing_demo.is_file():
    print('Giới hạn phạm vi: chưa có data/raw/global_electronics_retail_2019_2020.csv trong kỳ báo cáo này.')
else:
    print(f'Nguồn bổ sung kỳ này: {missing_demo.name}')


## 3. Kết quả chuẩn hoá fact đơn hàng

Quy tắc áp dụng cho mọi nguồn orders: strip tên cột và ánh xạ alias, yêu cầu đủ 10 cột nghiệp vụ, giữ đúng 10 cột + cột `Data Source` truy vết nguồn, chuẩn hoá `Order Date` và các giá trị số dạng `$1,200.50`, loại dòng thiếu trường chuẩn và dòng trùng, giữ lại dòng thỏa `Sales >= 0` và `Quantity > 0`, sắp xếp theo ngày. UNION dọc các nguồn và khử trùng liên nguồn khi cả 10 trường nghiệp vụ trùng nhau.

In [ ]:
order_sources = [
    ('global_superstore_orders.csv', 'Global Superstore (2012-2015)'),
    ('superstore_2015_2018.csv', 'Superstore (2015-2018)'),
    ('global_electronics_retail_2019_2020.csv', 'Kaggle Global Electronics Retail (demo, unverified)'),
    ('superstore_2021_2024.csv', 'Superstore (2021-2024)'),
]
cleaned_frames, skipped, before_after = [], [], []
for filename, source_name in order_sources:
    raw_path = RAW_DIR / filename
    if not raw_path.is_file():
        print(f'Nguồn chưa có trong kỳ báo cáo: {filename} | {source_name}')
        skipped.append(filename)
        continue
    with open(raw_path, encoding='utf-8', errors='ignore') as f:
        raw_lines = sum(1 for _ in f) - 1
    df = clean_orders(raw_path, TMP_DIR / f'clean_{filename}', source_name=source_name)
    cleaned_frames.append(df)
    before_after.append({'Data Source': source_name, 'Raw (ước lượng)': raw_lines, 'Sau clean': len(df), 'Loại bỏ': raw_lines - len(df)})
    print(f"{filename}: {raw_lines:,} → {len(df):,} dòng sau clean | ngày {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")

if not cleaned_frames:
    raise FileNotFoundError('Không còn file orders nào để UNION.')
orders_union = build_orders_union(cleaned_frames)
print(f'Fact sau làm sạch: {len(orders_union):,} dòng, {len(orders_union.columns)} cột')
display(pd.DataFrame(before_after))
source_summary = orders_union.groupby('Data Source', as_index=False).agg(
    Rows=('Order ID', 'size'), Orders=('Order ID', 'nunique'),
    Customers=('Customer ID', 'nunique'), Sales=('Sales', 'sum'), Profit=('Profit', 'sum')
)
display(source_summary)
fig = px.bar(source_summary, x='Data Source', y='Rows', color='Data Source', title='Quy mô fact theo nguồn dữ liệu')
fig.update_layout(showlegend=False, xaxis_title='Nguồn dữ liệu')
fig.show()
if skipped:
    print(f"Phạm vi kỳ báo cáo chưa gồm: {', '.join(skipped)}.")


## 4. Kết quả kiểm tra chất lượng fact

Bảng kiểm tra phản ánh hiện trạng sau làm sạch. Profit âm được giữ lại vì có thể là đơn lỗ thật. Không xóa trùng theo riêng `Order ID`, vì một đơn có thể có nhiều dòng sản phẩm.

In [ ]:
required_columns = [
    'Order ID', 'Order Date', 'Customer ID', 'Country', 'Region',
    'Category', 'Sub-Category', 'Sales', 'Quantity', 'Profit',
]
missing_contract_columns = [column for column in required_columns if column not in orders_union.columns]
quality_rows = [{'Kiểm tra': 'Thiếu cột theo hợp đồng', 'Số lượng': len(missing_contract_columns), 'Chi tiết': ', '.join(missing_contract_columns) or 'Không'}]
if not missing_contract_columns:
    parsed_order_dates = pd.to_datetime(orders_union['Order Date'], errors='coerce')
    quality_rows.extend([
        {'Kiểm tra': 'Giá trị thiếu toàn bảng', 'Số lượng': int(orders_union[required_columns].isna().sum().sum()), 'Chi tiết': 'Các trường thuộc schema chuẩn'},
        {'Kiểm tra': 'Order Date không parse được', 'Số lượng': int(parsed_order_dates.isna().sum()), 'Chi tiết': 'Order Date'},
        {'Kiểm tra': 'Sales < 0', 'Số lượng': int((pd.to_numeric(orders_union['Sales'], errors='coerce') < 0).sum()), 'Chi tiết': 'Theo hợp đồng hiện tại Sales >= 0'},
        {'Kiểm tra': 'Quantity <= 0', 'Số lượng': int((pd.to_numeric(orders_union['Quantity'], errors='coerce') <= 0).sum()), 'Chi tiết': 'Theo hợp đồng hiện tại Quantity > 0'},
        {'Kiểm tra': 'Profit < 0', 'Số lượng': int((pd.to_numeric(orders_union['Profit'], errors='coerce') < 0).sum()), 'Chi tiết': 'Không tự động xem là lỗi'},
        {'Kiểm tra': 'Dòng trùng hoàn toàn', 'Số lượng': int(orders_union.duplicated().sum()), 'Chi tiết': 'Không xóa trong notebook'},
    ])
quality_report = pd.DataFrame(quality_rows)
display(quality_report)
if missing_contract_columns:
    raise ValueError(f'Fact không đủ schema: {missing_contract_columns}')


## 5. Kết quả JOIN orders_enriched

Bảng People gồm 2 cột `Person, Region` sau khi chuẩn hoá. Fact được `LEFT JOIN` với người phụ trách theo `Region` (đổi `Person → Manager`). Region không có trong People thì `Manager` null — đơn hàng được giữ lại đầy đủ.

In [ ]:
people = clean_people(RAW_DIR / 'global_superstore_people.csv', TMP_DIR / 'people_cleaned.csv')
print(f'people_cleaned: {len(people):,} dòng, cột {list(people.columns)}')
display(people.head())

orders_enriched = join_orders_people(orders_union, people)
n_missing_mgr = int(orders_enriched['Manager'].isna().sum()) if 'Manager' in orders_enriched.columns else 0
print(f'orders_enriched: {len(orders_enriched):,} dòng (bằng fact) | Tỷ lệ thiếu Manager: {n_missing_mgr:,} ({n_missing_mgr / max(len(orders_enriched), 1):.1%})')
if 'Manager' in orders_enriched.columns:
    manager_coverage = orders_enriched.groupby('Region', dropna=False, as_index=False).agg(Rows=('Region', 'size'), Managers=('Manager', 'nunique'))
    display(manager_coverage.sort_values('Rows', ascending=False))
    print('Các Region thiếu Manager gồm Region Canada của Global và 4 Region Mỹ (Central/East/South/West) của nguồn 2021–2024.')


## 6. Kết quả làm sạch Walmart

Bảng stores chuẩn hoá kiểu và khử trùng theo `Store`. Bảng features chỉ loại dòng thiếu khóa `Store, Date` và dòng trùng khóa — các giá trị thiếu `Markdown/CPI/Unemployment` được giữ lại theo đúng đặc thù thu thập. Bảng train chỉ loại dòng thiếu khóa `Store, Dept, Date, Weekly_Sales` và dòng trùng khóa — giá trị `Weekly_Sales` âm được giữ lại. Bảng tổng hợp được `LEFT JOIN` theo `(Store, Date)` và `Store` nên giữ toàn bộ doanh số train.

In [ ]:
stores = clean_walmart_stores(RAW_DIR / 'walmart_stores.csv', TMP_DIR / 'walmart_stores_cleaned.csv')
features = clean_walmart_features(RAW_DIR / 'walmart_features.csv', TMP_DIR / 'walmart_features_cleaned.csv')
train = clean_walmart_train(RAW_DIR / 'walmart_train.csv', TMP_DIR / 'walmart_train_cleaned.csv')
print(f'stores: {len(stores):,} dòng | features: {len(features):,} dòng | train: {len(train):,} dòng')
print(f"Tỷ lệ thiếu MarkDown1 của features: {int(features['MarkDown1'].isna().sum()):,} dòng được giữ lại.")
print(f"Số dòng Weekly_Sales âm của train được giữ lại: {int((train['Weekly_Sales'] < 0).sum()):,}.")

walmart_enriched = join_walmart_weekly(train, features, stores)
print(f'walmart_weekly_enriched: {len(walmart_enriched):,} dòng (bằng train), {len(walmart_enriched.columns)} cột')
display(walmart_enriched.head(3))
store_sales = walmart_enriched.groupby('Store', as_index=False).agg(Weekly_Sales=('Weekly_Sales', 'sum'))
fig = px.bar(store_sales, x='Store', y='Weekly_Sales', title='Tổng Weekly_Sales theo cửa hàng Walmart')
fig.show()


## 7. Kết quả làm sạch BigMart

Cột `Item_Fat_Content` được gom về 2 giá trị (`Low Fat`, `Regular`). Giá trị `Item_Visibility == 0` được xử lý như thiếu và bù theo trung bình nhóm `Item_Identifier`. Giá trị thiếu `Item_Weight` được bù theo nhóm `Item`, phần còn lại bù theo trung vị toàn bảng. Giá trị thiếu `Outlet_Size` được bù theo mode nhóm `Outlet_Type`. Báo cáo bổ sung cột `Outlet_Age` mốc năm 2013.

In [ ]:
bigmart = clean_bigmart(RAW_DIR / 'bigmart_train.csv', TMP_DIR / 'bigmart_cleaned.csv')
print(f'bigmart_cleaned: {len(bigmart):,} dòng, {len(bigmart.columns)} cột, số ô thiếu: {int(bigmart.isna().sum().sum())}')
display(bigmart['Item_Fat_Content'].value_counts(dropna=False).rename_axis('Item_Fat_Content').reset_index(name='Rows'))
fig = px.bar(bigmart.groupby('Outlet_Type', as_index=False).agg(Item_Outlet_Sales=('Item_Outlet_Sales', 'sum')), x='Outlet_Type', y='Item_Outlet_Sales', title='BigMart: tổng Item_Outlet_Sales theo Outlet_Type')
fig.show()
display(bigmart.head(3))


## 8. Kết quả bảng RFM và doanh thu tháng

Hai bảng có grain riêng và không gộp vào fact. Bảng RFM tính từ fact (snapshot = ngày đơn mới nhất + 1) và bổ sung `Manager` theo `Region`. Bảng doanh thu tháng tổng hợp theo tháng và nguồn; tháng không có giao dịch không được tính thành doanh thu bằng 0.

In [ ]:
rfm_export = build_rfm_export(orders_union, people)
monthly = build_monthly_sales(orders_union)
print(f'rfm_customers: {len(rfm_export):,} khách hàng, các cột: {list(rfm_export.columns)}')
if 'Segment' in rfm_export.columns:
    segment_counts = rfm_export['Segment'].value_counts(dropna=False).rename_axis('Segment').reset_index(name='Customers')
    display(segment_counts)
    fig = px.bar(segment_counts, x='Segment', y='Customers', title='Số khách hàng theo phân khúc RFM')
    fig.show()
if {'Frequency', 'Monetary'}.issubset(rfm_export.columns):
    color_column = 'Segment' if 'Segment' in rfm_export.columns else None
    fig = px.scatter(rfm_export, x='Frequency', y='Monetary', color=color_column, title='Frequency và Monetary theo khách hàng')
    fig.show()
print(f'monthly_sales: {len(monthly):,} dòng (tháng x nguồn)')
display(monthly.head())
date_column = next((column for column in monthly.columns if 'date' in column.lower() or 'month' in column.lower()), None)
if date_column and 'Sales' in monthly.columns:
    plot_data = monthly.copy()
    plot_data[date_column] = pd.to_datetime(plot_data[date_column], errors='coerce')
    plot_data = plot_data.dropna(subset=[date_column]).sort_values(date_column)
    color_column = 'Data Source' if 'Data Source' in plot_data.columns else None
    fig = px.line(plot_data, x=date_column, y='Sales', color=color_column, title='monthly_sales: Sales theo tháng và nguồn', markers=True)
    fig.show()
    print('Lưu ý: các tháng không có nguồn trong bảng tổng hợp không phải doanh thu bằng 0.')


## 9. Kết luận và bàn giao

Kết quả sau làm sạch gồm: (1) quy mô raw và sau làm sạch theo từng nguồn; (2) fact đạt schema hợp đồng, Profit âm được giữ lại; (3) tỷ lệ `Manager` thiếu và các Region không khớp People; (4) giá trị thiếu đặc thù Walmart được giữ lại cùng `Weekly_Sales` âm; (5) BigMart đã xử lý thiếu và bổ sung `Outlet_Age`; (6) RFM và doanh thu tháng giữ đúng grain, không bù tháng trống thành 0.

Khoảng trống thời gian không được diễn giải là doanh thu bằng 0, không nội suy để lấp thiếu dữ liệu, và không kết luận nguyên nhân kinh doanh chỉ từ tương quan trong biểu đồ. Nguồn Kaggle minh họa khoảng 2019–2020 có provenance chưa xác minh; kết quả theo `Data Source` cần trích dẫn nguồn/license và không trình bày doanh số nguồn này như số liệu thực đã kiểm chứng.

Toàn bộ output được tái tạo bằng `python scripts/clean_data.py`; không sửa tay file trong `data/processed/`.